# Eksploracja: konwersacja z pełnym retrievalem (vision)

Sandbox dla wieloturowej konwersacji na ścieżce **vision** (WIP).
W każdej turze widzisz:
1. pytanie,
2. retrieved chunki (top_k) z `vision_vector_store`,
3. dokładnie to, co leci do modelu (system + kontekst + historia + pytanie),
4. odpowiedź modelu,
5. cited sources (chunki, na które wskazują `[Strona N]` w odpowiedzi).

Historia trzymana w `conversation.history` (in-memory dict) — przeżywa między komórkami,
gubi się przy restarcie kernela.

Notebook używa **tylko** vision (bez bridge z naive — naive wciąż obsługuje prod `/chat`).

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import os
import re
import sys
from collections import Counter
from pathlib import Path
from statistics import mean, median

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from backend.app.config import (
    CHAT_MODEL,
    VISION_CACHE_DIR,
    VISION_CHUNK_MAX_CHARS,
    VISION_CHUNK_OVERLAP_CHARS,
    VISION_TOP_K,
)
from backend.app.conversation.history import add_message, get_history, sessions
from backend.app.conversation.llm_client import ask
# from backend.app.conversation.prompt_builder import SYSTEM_PROMPT
from backend.app.document.models import (
    DocumentMetadata,
    ExtractedChapter,
    ExtractedDocument,
)
from backend.app.document.vision_chunker import chunk_document
from backend.app.retrieval import vision_vector_store as vvs

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("WARN: brak ANTHROPIC_API_KEY — `chat(...)` zwróci błąd z Anthropic API.")
else:
    print(f"OK: ANTHROPIC_API_KEY ustawiony. CHAT_MODEL = {CHAT_MODEL}")

OK: ANTHROPIC_API_KEY ustawiony. CHAT_MODEL = claude-sonnet-4-20250514


In [2]:
SYSTEM_PROMPT = (
    "Jesteś asystentem odpowiadającym na pytania wyłącznie na podstawie dostarczonych "
    "fragmentów dokumentu.\n\n"
    "Zasady:\n"
    "1. Odpowiadaj WYŁĄCZNIE na podstawie dostarczonych fragmentów. "
    "Nie korzystaj z wiedzy zewnętrznej.\n"
    "2. Każdą tezę potwierdź dosłownym cytatem z dokumentu w cudzysłowie.\n"
    "3. Przy każdym cytacie podaj numer strony lub zakres stron:\n"
    "   - Jeśli cytat pochodzi z jednej strony: [Strona X]\n"
    "   - Jeśli cytat obejmuje kilka kolejnych stron: [Strony X-Y]\n"
    "4. Fragmenty oznaczone jako TABELA lub INFOGRAFIKA traktuj jako dane pomocnicze.\n"
    "   NIE cytuj ich dosłownie. Zamiast tego opisz zawarte w nich informacje własnymi "
    "słowami i oznacz odwołanie w formacie:\n"
    "   - [Tabela, s. X] lub [Infografika, s. X]\n"
    "5. Jeśli informacji nie ma w dostarczonym kontekście — powiedz o tym wprost. "
    "Nie zgaduj i nie konfabuluj.\n"
    "6. Jeśli pytanie odwołuje się do wcześniejszej rozmowy (np. 'to', 'a rok wcześniej', "
    "'ten dokument'), wykorzystaj historię konwersacji do rozwiązania referencji.\n"
    "7. Odpowiadaj w tym samym języku, w którym zadano pytanie."
)

## 2. Parametry runu

- `CHAPTERS` — lista ID rozdziałów albo `"ALL"`.
- `PAGE_RANGES` — opcjonalny filtr stron: `"12"`, `"12-13"`, `"12-15,20,25-30"`.
- `MAX_CHARS`, `OVERLAP_CHARS` — z `config.py` (jedno źródło prawdy).
- `SESSION_ID` — stały, żeby kolejne `chat(...)` budowały jedną historię.

In [3]:
CHAPTERS: list[str] | str = "ALL"
# CHAPTERS = ["IV", "VI"]

PAGE_RANGES: str | None = None
# PAGE_RANGES = "51-52"

MAX_CHARS = VISION_CHUNK_MAX_CHARS          # 800
OVERLAP_CHARS = VISION_CHUNK_OVERLAP_CHARS  # 0
TOP_K = VISION_TOP_K                        # 5

SESSION_ID = "nb-session-1"
MAX_CONTEXT_PREVIEW = 800

CACHE_DIR = VISION_CACHE_DIR
CHAPTER_ORDER = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X"]
AVAILABLE_CHAPTERS = [cid for cid in CHAPTER_ORDER if (CACHE_DIR / f"{cid}.json").exists()]
print(f"CACHE_DIR:          {CACHE_DIR}")
print(f"Dostępne rozdziały: {AVAILABLE_CHAPTERS}")
print(f"MAX_CHARS={MAX_CHARS}  OVERLAP_CHARS={OVERLAP_CHARS}  TOP_K={TOP_K}")

CACHE_DIR:          C:\mirror\zadanie\docquery\data\extraction_v2
Dostępne rozdziały: ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X']
MAX_CHARS=800  OVERLAP_CHARS=0  TOP_K=5


## 3. Load + chunk + index

In [4]:
def parse_page_ranges(spec: str | None) -> set[int] | None:
    if not spec:
        return None
    out: set[int] = set()
    for part in spec.split(","):
        part = part.strip()
        if not part:
            continue
        if "-" in part:
            a, b = part.split("-", 1)
            out.update(range(int(a), int(b) + 1))
        else:
            out.add(int(part))
    return out


def load_document(
    chapters: list[str] | str,
    page_filter: set[int] | None,
) -> ExtractedDocument:
    chapter_ids = AVAILABLE_CHAPTERS if chapters == "ALL" else list(chapters)
    loaded: list[ExtractedChapter] = []
    for ch_id in chapter_ids:
        path = CACHE_DIR / f"{ch_id}.json"
        ch = ExtractedChapter.load_json(path)
        if page_filter is not None:
            ch.pages = [p for p in ch.pages if p.page_num in page_filter]
        if ch.pages:
            loaded.append(ch)
    if not loaded:
        raise ValueError("Brak stron po filtrze — sprawdź CHAPTERS/PAGE_RANGES.")
    total_pages = sum(len(ch.pages) for ch in loaded)
    return ExtractedDocument(
        metadata=DocumentMetadata(
            source_file="raport_2024_pl.pdf",
            total_pages=total_pages,
            extraction_date="notebook",
        ),
        title="BGK 2024 v2 (podzbiór)",
        chapters=loaded,
    )


doc = load_document(CHAPTERS, parse_page_ranges(PAGE_RANGES))
chunks = chunk_document(doc, max_chars=MAX_CHARS, overlap_chars=OVERLAP_CHARS)

lengths = [len(c["content"]) for c in chunks]
print(f"Rozdziały:       {[ch.chapter_id for ch in doc.chapters]}")
print(f"Stron:           {len(doc.get_all_pages())}")
print(f"Chunków:         {len(chunks)}")
print(f"Typy chunków:    {dict(Counter(c['element_type'] for c in chunks))}")
if lengths:
    print(
        f"Długość content: min={min(lengths)}  median={int(median(lengths))}  "
        f"mean={int(mean(lengths))}  max={max(lengths)}"
    )

vvs.reset_collection()
vvs.index_vision_chunks(chunks)
print(f"Zaindeksowano {len(chunks)} chunków.")

Rozdziały:       ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X']
Stron:           182
Chunków:         802
Typy chunków:    {'text': 650, 'infographic': 44, 'table': 108}
Długość content: min=26  median=645  mean=679  max=3731
Zaindeksowano 802 chunków.


## 4. Formatowanie kontekstu + `chat(...)`

`build_prompt` z `prompt_builder.py` zakłada naive shape (`text`, `page`). Vision chunki mają
`content`, `pages`, `chapter`, `section`, `element_type` — więc budujemy własny formatter
i składamy `messages` ręcznie, reużywając tylko `SYSTEM_PROMPT` (ta sama polityka no-hallucination).

In [5]:
SEP = "─" * 80

_CITATION_RE = re.compile(
    r"\[(?:"
    r"Strona\s+(?P<page>\d+)"
    r"|Strony\s+(?P<p_start>\d+)\s*[-–]\s*(?P<p_end>\d+)"
    r"|Tabela,\s*s\.\s*(?P<t_page>\d+)"
    r"|Infografika,\s*s\.\s*(?P<i_page>\d+)"
    r")\]"
)


def _parse_citations(answer: str) -> list[tuple[str, set[int]]]:
    """Wyciąga (element_type, {pages}) z każdego [Strona…]/[Strony…]/[Tabela…]/[Infografika…]."""
    out: list[tuple[str, set[int]]] = []
    for m in _CITATION_RE.finditer(answer):
        if m.group("page"):
            out.append(("text", {int(m.group("page"))}))
        elif m.group("p_start"):
            s, e = int(m.group("p_start")), int(m.group("p_end"))
            out.append(("text", set(range(s, e + 1))))
        elif m.group("t_page"):
            out.append(("table", {int(m.group("t_page"))}))
        elif m.group("i_page"):
            out.append(("infographic", {int(m.group("i_page"))}))
    return out


def _match_sources(
    retrieved: list[dict],
    citations: list[tuple[str, set[int]]],
) -> list[dict]:
    """Unikatowe (po chunk_index) chunki pasujące do cytatów — match po (element_type, page)."""
    seen: set[int] = set()
    sources: list[dict] = []
    for c in retrieved:
        for etype, pages in citations:
            if c["element_type"] != etype:
                continue
            if any(p in pages for p in c["pages"]):
                key = c["chunk_index"]
                if key not in seen:
                    seen.add(key)
                    sources.append(c)
                break
    return sources


def format_vision_context(chunks: list[dict]) -> str:
    parts = []
    for c in chunks:
        header_bits = [x for x in (c["chapter"], c["section"]) if x]
        header = " › ".join(header_bits)
        pages_str = (
            f"Strona {c['page']}" if len(c["pages"]) == 1
            else f"Strony {c['pages'][0]}–{c['pages'][-1]}"
        )
        parts.append(
            f"[{pages_str}] ({c['element_type']}) {header}\n\"{c['content']}\""
        )
    return "\n\n".join(parts)


def _short(text: str, limit: int = 300) -> str:
    text = text.replace("\n", " ").strip()
    return text if len(text) <= limit else text[:limit] + "…"


def _render_turn(question, retrieved, messages, answer, sources):
    print(SEP)
    print(f"QUERY: {question}")
    print(SEP)
    print(f"RETRIEVED (top_k={len(retrieved)}):")
    for i, r in enumerate(retrieved, 1):
        header = f"{r['chapter'] or '-'} › {r['section'] or '-'}"
        print(
            f"  [{i}] {header}  pages={r['pages']}  "
            f"type={r['element_type']}  dist={r['distance']:.3f}"
        )
        preview = r["content"][:MAX_CONTEXT_PREVIEW].strip()
        print("      " + preview.replace("\n", "\n      "))
        if len(r["content"]) > MAX_CONTEXT_PREVIEW:
            print("      …")
        print()
    print(SEP)
    context_len = len(messages[0]["content"])
    history_msgs = len(messages) - 2  # minus context and final question
    print("PROMPT SENT:")
    print(f"  system:       {_short(SYSTEM_PROMPT, 120)}")
    print(f"  context:      {context_len} znaków (pierwsza user-message)")
    print(f"  history msgs: {history_msgs}")
    for i, m in enumerate(messages[1:-1], 1):
        print(f"    [{i}] {m['role']:<9} {_short(m['content'], 200)}")
    print(f"  question:     {messages[-1]['content']}")
    print(SEP)
    print("ANSWER:")
    print(answer)
    print(SEP)
    print(f"CITED SOURCES ({len(sources)}):")
    for s in sources:
        header = f"{s['chapter'] or '-'} › {s['section'] or '-'}"
        print(f"  - pages={s['pages']} type={s['element_type']}  {header}")
    print()


def chat(
    question: str,
    session_id: str = SESSION_ID,
    top_k: int = TOP_K,
    *,
    show: bool = True,
) -> dict:
    retrieved = vvs.search_vision(question, top_k=top_k)
    history = get_history(session_id)

    context = format_vision_context(retrieved)
    messages = [
        {"role": "user", "content": f"Document context:\n\n{context}"},
        *history,
        {"role": "user", "content": question},
    ]

    answer = ask(messages, SYSTEM_PROMPT)

    add_message(session_id, "user", question)
    add_message(session_id, "assistant", answer)

    citations = _parse_citations(answer)
    sources = _match_sources(retrieved, citations)

    if show:
        _render_turn(question, retrieved, messages, answer, sources)

    return {
        "retrieved": retrieved,
        "messages": messages,
        "answer": answer,
        "sources": sources,
    }


def show_history(session_id: str = SESSION_ID) -> None:
    h = get_history(session_id)
    print(f"Historia '{session_id}' — {len(h)} wiadomości:")
    for i, m in enumerate(h, 1):
        print(f"  [{i}] {m['role']:<9} {_short(m['content'], 200)}")


def reset_session(session_id: str = SESSION_ID) -> None:
    sessions.pop(session_id, None)
    print(f"Sesja '{session_id}' wyczyszczona.")

## 5. Przykładowa konwersacja

Kolejne tury budują się na poprzednich — testują rozwiązywanie referencji ("a ile",
"w tym samym roku", "w jakim procencie") z historii.

In [6]:
reset_session()
last = chat("Kiedy uruchomiono Forum Klienta?")

Sesja 'nb-session-1' wyczyszczona.
────────────────────────────────────────────────────────────────────────────────
QUERY: Kiedy uruchomiono Forum Klienta?
────────────────────────────────────────────────────────────────────────────────
RETRIEVED (top_k=5):
  [1] IV Otoczenie › 3. Doświadczenia klientów  pages=[51]  type=text  dist=0.443
      W Departamencie Zarządzania Doświadczeniami Klienta regularnie organizujemy spotkania Komitetu CX. Analizujemy zgromadzony głos klientów z różnych źródeł i najważniejsze czynniki, które wpływają na ich satysfakcję. Przygotowujemy rekomendacje zmian, przekazujemy je do właściwych jednostek banku i planujemy inicjatywy proklienckie.
      
      W 2024 roku uruchomiliśmy Forum Klienta. Są to spotkania, podczas których analizujemy z właścicielami biznesowymi główne punkty niezadowolenia naszych klientów, przygotowujemy rekomendacje oraz sposób i harmonogram wprowadzania zmian.
      
      Budujemy kulturę klientocentryczną

  [2] IV Otoczenie › 3. 

In [9]:
show_history()

Historia 'nb-session-1' — 6 wiadomości:
  [1] user      Kiedy uruchomiono Forum Klienta?
  [2] assistant Na podstawie dostarczonych fragmentów dokumentu, Forum Klienta zostało uruchomione w 2024 roku. Dokument stwierdza: "W 2024 roku uruchomiliśmy Forum Klienta. Są to spotkania, podczas których analizuje…
  [3] user      A ile reklamacji wpłynęło do banku w tym samym roku?
  [4] assistant Na podstawie dostarczonych fragmentów dokumentu, w 2024 roku do banku wpłynęło łącznie 333 skargi i reklamacje. Dokument podaje: "Łącznie w 2024 r. do banku wpłynęły 333 skargi i reklamacje, z czego 3…
  [5] user      W jakim procencie zostały uznane za zasadne?
  [6] assistant Na podstawie dostarczonych fragmentów dokumentu, w 2024 roku 35% skarg i reklamacji zostało uznanych za zasadne. Jak stwierdza dokument: "Łącznie w 2024 r. do banku wpłynęły 333 skargi i reklamacje, z…


## 6. Własny sandbox

Pisz własne pytania niżej. `last` trzyma ostatni wynik — możesz klikać
`last["retrieved"]`, `last["messages"]`, `last["sources"]` w osobnych komórkach.

In [10]:
# last = chat("...")

In [14]:
reset_session()

Sesja 'nb-session-1' wyczyszczona.


In [15]:
last = chat("Ile nowych aktów prawnych zidentyfikowano jako wymagające implementacji w 2024 r. w porównaniu do 2023 r.?")

────────────────────────────────────────────────────────────────────────────────
QUERY: Ile nowych aktów prawnych zidentyfikowano jako wymagające implementacji w 2024 r. w porównaniu do 2023 r.?
────────────────────────────────────────────────────────────────────────────────
RETRIEVED (top_k=5):
  [1] VI Ład korporacyjny › 2. Zmiany w otoczeniu regulacyjnym  pages=[106]  type=table  dist=0.231
      | Tabela 59. Zmiany w prawie
      
      Tabela przedstawiająca zmiany w prawie w latach 2024 i 2023.
      Liczba aktów prawnych w poszczególnych kategoriach.
      
      | | 2024 | 2023 |
      |---|---|---|
      | Akty prawne zaimplementowane | 35 | 52 |
      | Akty prawne zidentyfikowane jako wymagające implementacji (nowe) | 18 | 11 |
      | Akty prawne w trakcie implementacji | 19 | 24 |

  [2] X Załączniki › 6. Zmiany otoczenia regulacyjnego  pages=[178]  type=text  dist=0.307
      - Ustawa z dnia 20 marca 2024 r. o zmianie ustawy - Kodeks cywilny, ustawy o kredycie konsumencki